# Training from a saved collection

A **collection** is a group of datasets, saved in the neuralshed database, that were assembled to train one decoder together. It records three things: which datasets are members, the single `decode_target` they share, and the `CombineStudies` step chain that pools them into one event table.

Collections are normally composed in the collection builder app and then loaded from code. This notebook walks the shortest path from a name in the database to a pandas DataFrame of events:

```python
from neuralshed.database import collections
events = collections.load_collection("mycollection").run()
```

The sections below unpack that one-liner: browse what is saved, look at the stored config, turn it into a runnable `Step`, run it, and read the frame that comes out.

In [1]:
import warnings; warnings.filterwarnings("ignore")

import json
import pandas as pd

from neuralshed.database import collections

pd.set_option("display.width", 160, "display.max_columns", 40)

## 1. Browse the saved collections

`list_collections` returns one row per collection, with its member datasets rolled up into a single column.

In [2]:
collections.list_collections()

,id,name,decode_target,n_datasets,datasets
0,1,mycollection,text,3,"card_2024, fan_h2, willet_2023"
1,2,testing,text,3,"card_2024, fan_h2, willet_2023"
2,3,testing2,text,3,"card_2024, fan_h2, willet_2023"


## 2. Inspect one collection

`get_collection` returns the full record. Each member carries a coarse `region` label, which is what the region embedding keys off, and every member shares the collection's one `decode_target` (one decoder head).

In [3]:
NAME = "mycollection"
col = collections.get_collection(NAME)

print("decode_target:", col["decode_target"])
pd.DataFrame(col["datasets"])

decode_target: text


,dataset_id,region
0,card_2024,HOMBA_AA30009
1,fan_h2,HOMBA_10165
2,willet_2023,HOMBA_10161


The `study` field holds the composed step chain as plain JSON. It is a `Chain` headed by `CombineStudies`, whose `chains` map one member key to that dataset's `Study`, optionally followed by per-study transforms. Steps placed *after* the fan-in apply to the pooled collection instead.

Notice that no `path` appears anywhere. The stored config is machine-independent, and the data directories are filled in at load time.

In [4]:
print(json.dumps(col["study"], indent=2))

{
  "name": "Chain",
  "steps": {
    "CombineStudies": {
      "name": "CombineStudies",
      "chains": {
        "Card2024Brain2Text_DummyTransform": [
          {
            "name": "Card2024Brain2Text"
          },
          {
            "name": "DummyTransform",
            "note": "test"
          }
        ],
        "FanFalconH2": [
          {
            "name": "FanFalconH2"
          }
        ],
        "Willett2023Sentences": [
          {
            "name": "Willett2023Sentences"
          }
        ]
      }
    }
  }
}


## 3. Instantiate the Step

`ns.Step` resolves each step from its `name` discriminator against the classes that have registered themselves, so a stored config only loads once those classes are imported. `load_collection` handles that and two other things the bare config cannot do on its own:

1. imports `neuralshed.studies` and `neuralshed.transforms`, so every `name` in the config resolves,
2. looks up each member's data directory on this machine (`data_dirs`),
3. injects those paths into a copy of the config (`inject_paths`) and passes the result to `ns.Step`.

`must_exist=True` makes it fail up front with a clear error when the data drive is not mounted, rather than partway through the run.

In [5]:
step = collections.load_collection(NAME, must_exist=True)
type(step)

neuralset.base.Chain

These are the paths that were injected. A config names its members by study class, so the mapping is keyed on the class name rather than the dataset id.

In [6]:
dataset_ids = [m["dataset_id"] for m in col["datasets"]]
collections.data_dirs(dataset_ids)

{'Card2024Brain2Text': PosixPath('/Volumes/seagate_0/tether/datasets/card_2024/data'),
 'FanFalconH2': PosixPath('/Volumes/seagate_0/tether/datasets/fan_h2/data'),
 'Willett2023Sentences': PosixPath('/Volumes/seagate_0/tether/datasets/willet_2023/data')}

## 4. Run it

`step.run()` runs each member chain and lets `CombineStudies` concatenate the results into one standardized event table. Members are concatenated in a canonical order (by study class name, then key), so the output depends only on what the config says.

This reads the source files, so expect roughly a minute for this collection.

In [7]:
events = step.run()
events.shape

(43117, 32)

Thirty-odd columns, because the union of what three studies annotate is wider than what any one of them uses. Columns a study does not populate come back as `NaN` for its rows.

In [8]:
list(events.columns)

['type',
 'start',
 'duration',
 'timeline',
 'subject',
 'session',
 'task',
 'run',
 'feature',
 'unit',
 'frequency',
 'filepath',
 'method',
 'threshold',
 'text',
 'language',
 'modality',
 'phonemes',
 'study',
 'original_split',
 'intrasession_trial',
 'block_num',
 'task_condition',
 'lower_freq',
 'upper_freq',
 'context',
 'stop',
 'cue_raw',
 'speaking_mode',
 'delay_time',
 'loop_mode',
 'block_type']

One row per event. `type` says what kind of event it is, `study` says which member produced it, and `timeline` names the continuous recording segment the event sits on.

In [9]:
preview = ["type", "study", "subject", "session", "start", "duration", "text"]
events[preview].head(8)

,type,study,subject,session,start,duration,text
0,SpikeCounts,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,6.42,NaN
1,BandPower,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,6.42,NaN
2,Sentence,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,6.42,Bring it closer.
3,SpikeCounts,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,9.62,NaN
4,BandPower,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,9.62,NaN
5,Sentence,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,9.62,My family is closer.
6,SpikeCounts,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,9.60,NaN
7,BandPower,Card2024Brain2Text,Card2024Brain2Text/T15,t15.2023.08.11,0.0,9.60,NaN


## 5. Reading the pooled table

Counting events by member and type shows the heterogeneity that `CombineStudies` exists to absorb.

In [10]:
events.pivot_table(index="study", columns="type", values="start", aggfunc="size", fill_value=0)

type,BandPower,Sentence,SpikeCounts
study,,,
Card2024Brain2Text,10948,9498,10948
FanFalconH2,0,728,47
Willett2023Sentences,24,10900,24


The three members do not agree on much:

- **Card2024Brain2Text** slices one timeline per trial, so its neural features arrive one event per trial. Only 9498 of those 10948 trials carry a `Sentence`, the rest are unlabelled.
- **Willett2023Sentences** and **FanFalconH2** keep one timeline per session. A single long neural-feature event spans the whole session, and many `Sentence` events sit inside it.
- **FanFalconH2** ships no `BandPower` at all, only threshold crossings.

The per-study view below makes the timeline granularity explicit: 10948 timelines across 45 card sessions (one per trial), against 24 and 47 timelines for the two session-level studies.

In [11]:
events.groupby("study").agg(
    sessions=("session", "nunique"),
    timelines=("timeline", "nunique"),
    events=("type", "size"),
    sentences=("type", lambda s: (s == "Sentence").sum()),
)

,sessions,timelines,events,sentences
study,,,,
Card2024Brain2Text,45,10948,31394,9498
FanFalconH2,47,47,775,728
Willett2023Sentences,24,24,10948,10900


## Where this goes next

The frame above is the raw material for a decoder, not yet a training set. Turning it into fixed-length clips means windowing every `Sentence` against the neural features on its timeline, z-scoring per session, and building a shared character vocabulary. That is what [`01_supervised_multitask.ipynb`](01_supervised_multitask.ipynb) and [`02_pretrain_finetune.ipynb`](02_pretrain_finetune.ipynb) do with `build_pooled_dataset`, which currently composes the same fan-in from a hardcoded `NEURAL_TEXT_COLLECTION` spec list.

A database collection is the persisted, UI-composed form of that spec list. Loading one gives you the pooled event table without hardcoding which studies, filters, or transforms went into it.